# Multi-Agency Ridership Data Standardization and Stop-Level Aggregation Workflow


This notebook processes ridership data from multiple transit agencies to create representative stop-level datasets for weekdays, Saturdays, and Sundays. For each agency, the workflow generally includes:

- Adding day type based on the service date (Weekday, Saturday, Sunday), with holidays normalized or removed as needed.
- Cleaning and renaming columns to ensure consistency across datasets (stop IDs, stop names, route names, and ridership fields).
- Summing boardings and alightings across directions at each stop, and aggregating multiple time periods where applicable.
- Calculating weighted averages when ridership is reported over multiple periods, using the number of service days as weights.
- Setting a ridership basis flag to indicate that values represent reported daily boardings and/or alightings.
- Computing final average ridership per stop, route, and day type, producing clean, representative stop-level datasets ready for modeling, analysis, or comparison across agencies.

This approach standardizes ridership data across agencies with differing data formats, missing values, and service coverage, enabling consistent cross-agency analysis.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [6]:
# Counting number of days without holidays
def count_service_days(row):
    
    if str(row.day_type).lower() == "all":
        return (row.end_date - row.start_date).days + 1

    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)

    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)

    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [7]:
# Counting number of days with holidays
def count_service_days_wo_holidays(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum((d.weekday() < 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Saturday":
        return sum((d.weekday() == 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Sunday":
        return sum((d.weekday() == 6) and (d not in holiday_set) for d in dates)

In [8]:
def compute_averages(df, ridership_cols, group_cols):
    """
    Computes average of one or more ridership columns per group in long format.

    Parameters:
    - df: pandas DataFrame
    - ridership_cols: list of columns to average (e.g., ['daily_boardings', 'daily_alightings'])
    - group_cols: list of columns to group by (e.g., ['stop_name', 'day_type'])

    Returns:
    - DataFrame with columns: group_cols + average of each ridership column
    """
    
    if isinstance(ridership_cols, str):
        ridership_cols = [ridership_cols]  # allow single string input
    
    averaged = df.groupby(group_cols)[ridership_cols].mean().reset_index()
    
    # rename each column to indicate it's averaged
    averaged = averaged.rename(columns={col: f"average_{col}" for col in ridership_cols})
    
    return averaged

In [9]:
master_cols = [
    "organization_name",
    "route_id",
    "route_name",
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "day_type",
    "average_daily_boardings",
    "average_daily_alightings",
    "start_date",
    "end_date"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

In [10]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')

bart_ridership = bart_ridership.rename(columns={
    'Station Name': 'stop_name',
    'Stop ID': 'stop_id',
    'Date': 'date'
})

# Grouping columns
group_cols = [
    "date", "stop_id", "stop_name",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_bart_ridership = (
    bart_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("Entries", "sum"),
        daily_alightings=("Exits", "sum")
    )
)

# Set the ridership basis flag
total_bart_ridership["daily_ridership_basis"] = "reported_daily"


In [11]:
# Calculate average ridership per day type and stop
total_bart_ridership['daily_boardings'] = pd.to_numeric(total_bart_ridership['daily_boardings'], errors='coerce')
total_bart_ridership['daily_alightings'] = pd.to_numeric(total_bart_ridership['daily_alightings'], errors='coerce')

average_bart_ridership = compute_averages(
    df=total_bart_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "day_type", "stop_name"]
)

average_bart_ridership["organization_name"] = "San Francisco Bay Area Rapid Transit District"

average_bart_ridership[['start_date','end_date']] = [bart_ridership['start_date'].min(), bart_ridership['end_date'].max()]
# Remove last digit from all stop_id
average_bart_ridership['stop_id'] = average_bart_ridership['stop_id'].astype(str).str[:-1]

bart_final = standardize_columns(average_bart_ridership, master_cols)

bart_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
92,San Francisco Bay Area Rapid Transit District,None,None,90410,Ashby,None,None,Weekday,2038.628352,1925.000000,2024-10-01,2025-09-30
31,San Francisco Bay Area Rapid Transit District,None,None,90170,Glen Park,None,None,Sunday,1313.826923,1382.596154,2024-10-01,2025-09-30
29,San Francisco Bay Area Rapid Transit District,None,None,90160,24th Street Mission,None,None,Weekday,5564.528736,5441.310345,2024-10-01,2025-09-30
121,San Francisco Bay Area Rapid Transit District,None,None,90620,South San Francisco,None,None,Sunday,463.615385,486.596154,2024-10-01,2025-09-30
16,San Francisco Bay Area Rapid Transit District,None,None,90120,Montgomery Street,None,None,Sunday,3031.538462,3516.865385,2024-10-01,2025-09-30


In [12]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'route_number': 'route_id',
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type','route_id','route_name',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)


In [13]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type','route_id','route_name', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_total_big_blue_bus_ridership["organization_name"] = "Big Blue Bus"

averaged_total_big_blue_bus_ridership[['start_date','end_date']] = [big_blue_bus_ridership['start_date'].min(), big_blue_bus_ridership['end_date'].max()]

big_blue_bus_ridership_final = standardize_columns(averaged_total_big_blue_bus_ridership, master_cols)
big_blue_bus_ridership_final.sample(5)

/tmp/ipykernel_1813/404159155.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1743,Big Blue Bus,1,Main St & Santa Monica Blvd/UCLA,1382,SANTA MONICA EB/YALE FS,34.035713,-118.471162,Weekday,35.602283,39.687527,2024-08-01,2025-11-30
1668,Big Blue Bus,43,San Vicente Blvd & 26th St,3136,PICO EB/17TH FS,34.017909,-118.472244,Sunday,6.200000,20.600000,2024-08-01,2025-11-30
1950,Big Blue Bus,5,Olympic Blvd,1179,COLORADO WB/14TH NS,34.021604,-118.482519,Weekday,0.378419,2.754979,2024-08-01,2025-11-30
1428,Big Blue Bus,14,Bundy Dr & Centinela Ave,2563,CENTINELA SB/PALMS NS,34.010689,-118.439070,Sunday,2.699539,4.444940,2024-08-01,2025-11-30
602,Big Blue Bus,14,Bundy Dr & Centinela Ave,2213,MONTANA EB/BUNDY FS,34.050725,-118.472219,Saturday,0.870624,24.691757,2024-08-01,2025-11-30


In [14]:
caltrain_ridership = caltrain_ridership.rename(columns={
    'Origin Station': 'stop_name',
    'Date Type': 'day_type',
    'Average Ridership':'average_daily_boardings_period',  
})

caltrain_ridership = caltrain_ridership[
    caltrain_ridership["day_type"] != "Holiday"
]

# Getting number of days and using US Federal Holidays
start_date_caltrain = '2023-11-01'
end_date_caltrain = '2025-07-31'

#Getting all the federal holidays between the start and end date
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=start_date_caltrain, end=end_date_caltrain)

# Setting holidays to calculate the number of weekdays, saturday and sunday without holidays 
holiday_set = set(holidays)

caltrain_ridership["start_date"] = pd.to_datetime(caltrain_ridership["start_date"])
caltrain_ridership["end_date"] = pd.to_datetime(caltrain_ridership["end_date"])

caltrain_ridership["n_days"] = caltrain_ridership.apply(count_service_days_wo_holidays, axis=1)

In [15]:
# Taking weighted average of the ridership across 20 months time periods
averaged_caltrain_ridership = (
    caltrain_ridership
    .groupby(['day_type','stop_name'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days'])
    }))
    .reset_index()
)

# Set the ridership basis flag
averaged_caltrain_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_caltrain_ridership["organization_name"] = "Caltrain"

averaged_caltrain_ridership[['start_date','end_date']] = [caltrain_ridership['start_date'].min(), caltrain_ridership['end_date'].max()]

caltrain_ridership_final = standardize_columns(averaged_caltrain_ridership, master_cols)

caltrain_ridership_final.sample(5)

/tmp/ipykernel_1813/3722569284.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
5,Caltrain,None,None,None,Burlingame,None,None,Saturday,324.633335,None,2023-11-01,2025-07-31
62,Caltrain,None,None,None,Belmont,None,None,Weekday,543.256104,None,2023-11-01,2025-07-31
79,Caltrain,None,None,None,San Antonio,None,None,Weekday,537.773367,None,2023-11-01,2025-07-31
75,Caltrain,None,None,None,Morgan Hill,None,None,Weekday,103.849297,None,2023-11-01,2025-07-31
65,Caltrain,None,None,None,Burlingame,None,None,Weekday,542.005575,None,2023-11-01,2025-07-31


In [16]:
culver_citybus_ridership["daily_ridership_basis"] = "reported_avg_daily"
culver_citybus_ridership["Route"] = culver_citybus_ridership["Route"].str.split("-", n=1).str[1]


culver_citybus_ridership = culver_citybus_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'route_id': 'route_id',
    'Route': 'route_name',
})

group_cols = [
    "route_name", "stop_id", "stop_name", "route_id", "day_type", "daily_ridership_basis"]

# Sum AVG On and AVG Off
sum_culver_citybus_ridership = (culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("AVG On", "sum"),
        average_daily_alightings=("AVG Off", "sum")
    )
)

sum_culver_citybus_ridership["organization_name"] = "Culver City Bus"

sum_culver_citybus_ridership[['start_date','end_date']] = [culver_citybus_ridership['start_date'].min(), culver_citybus_ridership['end_date'].max()]

culver_citybus_final = standardize_columns(sum_culver_citybus_ridership, master_cols)

culver_citybus_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
708,Culver City Bus,Rapid 6,Sepulveda Boulevard,665,SepulvedaBlvd/WashingtonBlvd,None,None,Weekday,58.9,55.6,2025-07-14,2025-08-25
259,Culver City Bus,3,Crosstown,904,Santa Monica Blvd/Ave of the Stars,None,None,Weekday,20.0,10.1,2025-07-14,2025-08-25
902,Culver City Bus,1,Washington Boulevard,138,Washington Blvd/Roberts Ave,None,None,Saturday,0.0,8.5,2025-07-14,2025-08-25
160,Culver City Bus,3,Crosstown,363,Motor Ave/Tabor St,None,None,Weekday,10.3,12.0,2025-07-14,2025-08-25
930,Culver City Bus,1,Washington Boulevard,147,ELineCulverCityStation,None,None,Sunday,128.1,5.9,2025-07-14,2025-08-25


In [17]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

foothill_transit_ridership = foothill_transit_ridership.rename(columns={
    'route_short_name': 'route_id',
    'stop_code': 'stop_id',
})

# Grouping columns
group_cols = [
    "date", "route_id", "stop_id", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_ridership_foothill['daily_boardings'] = pd.to_numeric(total_ridership_foothill['daily_boardings'], errors='coerce')
total_ridership_foothill['daily_alightings'] = pd.to_numeric(total_ridership_foothill['daily_alightings'], errors='coerce')

average_foothill_ridership = compute_averages(
    df=total_ridership_foothill,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", 
                "stop_lat", "stop_lon", "day_type"]
)

average_foothill_ridership["organization_name"] = "Foothill Transit"

average_foothill_ridership[['start_date','end_date']] = [foothill_transit_ridership['start_date'].min(), foothill_transit_ridership['end_date'].max()]
foothill_final = standardize_columns(average_foothill_ridership, master_cols)

foothill_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3966,Foothill Transit,285,None,1381,None,34.000274,-117.931110,Sunday,7.294118,5.000000,2024-07-01,2025-06-30
3616,Foothill Transit,282,None,2962,None,34.072325,-118.033712,Saturday,2.657143,0.942857,2024-07-01,2025-06-30
3025,Foothill Transit,280,None,777,None,34.118289,-117.907800,Weekday,19.743295,5.996169,2024-07-01,2025-06-30
5601,Foothill Transit,488,None,1354,None,34.071095,-117.977677,Saturday,13.307692,11.846154,2024-07-01,2025-06-30
1857,Foothill Transit,194,None,3389,None,34.023287,-117.956149,Weekday,10.707692,3.730769,2024-07-01,2025-06-30


In [18]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

# Calculate average ridership per day type and stop
fresno_area_express_ridership['daily_boardings'] = pd.to_numeric(fresno_area_express_ridership['daily_boardings'], errors='coerce')
fresno_area_express_ridership['daily_alightings'] = pd.to_numeric(fresno_area_express_ridership['daily_alightings'], errors='coerce')

average_fresno_area_express_ridership = compute_averages(
    df=fresno_area_express_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id",  "stop_name", "day_type"]
)

average_fresno_area_express_ridership["organization_name"] = "Fresno County"

average_fresno_area_express_ridership[['start_date','end_date']] = [fresno_area_express_ridership['start_date'].min(), fresno_area_express_ridership['end_date'].max()]

fresno_final = standardize_columns(average_fresno_area_express_ridership, master_cols)

fresno_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2362,Fresno County,None,None,1133,NW FRESNO - FWY 180,None,None,Sunday,7.899328,5.852286,2024-09-01,2025-08-31
3889,Fresno County,None,None,1806,NW ASHLAN - WINERY,None,None,Sunday,3.879035,0.831883,2024-09-01,2025-08-31
3429,Fresno County,None,None,1600,SE SHIELDS - MILLBROOK,None,None,Saturday,6.385911,7.973087,2024-09-01,2025-08-31
1456,Fresno County,None,None,711,SW PALM - SAN JOSE,None,None,Saturday,0.804275,1.288541,2024-09-01,2025-08-31
2992,Fresno County,None,None,1404,SW WALNUT - GEARY,None,None,Sunday,3.823899,6.869617,2024-09-01,2025-08-31


In [19]:
gold_coast_transit_ridership = gold_coast_transit_ridership.rename(columns={
    'day_of_week': 'day_type',
    'route': 'route_id',
    'lat': 'stop_lat',
    'lon': 'stop_lon'
})

group_cols = ["day_type", "route_id", "stop_id", "stop_name",  "stop_lat", "stop_lon", "start_date", "end_date"]

# Sum AVG On 
total_gold_coast_transit_ridership = (gold_coast_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("total_on", "sum"),
        daily_alightings=("total_off", "sum"),
    )
)

total_gold_coast_transit_ridership['start_date'] = pd.to_datetime(total_gold_coast_transit_ridership['start_date'])
total_gold_coast_transit_ridership['end_date'] = pd.to_datetime(total_gold_coast_transit_ridership['end_date'])

total_gold_coast_transit_ridership["n_days"] = total_gold_coast_transit_ridership.apply(count_service_days, axis=1)

In [20]:
# Taking weighted average of the ridership across 4 different time periods
averaged_gold_coast_transit_ridership = (
    total_gold_coast_transit_ridership
    .groupby(['day_type','route_id', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['daily_boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['daily_alightings'], weights=x['n_days'])
    }))
    .reset_index()
)

averaged_gold_coast_transit_ridership["organization_name"] = "Gold Coast Transit"

averaged_gold_coast_transit_ridership[['start_date','end_date']] = [gold_coast_transit_ridership['start_date'].min(), gold_coast_transit_ridership['end_date'].max()]
golden_coast_transit_final = standardize_columns(averaged_gold_coast_transit_ridership, master_cols)

golden_coast_transit_final.sample(5)

/tmp/ipykernel_1813/93958713.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
666,Gold Coast Transit,21,None,VICRAL1,Victoria & Ralston,34.261264,-119.211414,Weekday,9.0,10.0,2025-05-01,2025-05-31
284,Gold Coast Transit,8,None,WESROS2,Westar & Rose,34.186262,-119.160685,Weekday,3.0,2.0,2025-05-01,2025-05-31
678,Gold Coast Transit,23,None,HUECRT1,Hueneme & Courtland,34.147583,-119.179841,Weekday,8.0,17.0,2025-05-01,2025-05-31
283,Gold Coast Transit,8,None,WESROS1,Westar & Rose,34.186091,-119.160384,Weekday,3.0,1.0,2025-05-01,2025-05-31
478,Gold Coast Transit,16,None,THOCHE2,Thompson & Chestnut,34.278187,-119.290992,Weekday,13.0,7.0,2025-05-01,2025-05-31


In [21]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "Stop ID", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date',
    'Stop ID': 'stop_id'
})

# Calculate average ridership per day type and stop
total_golden_gate_park_shuttle_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_park_shuttle_ridership['daily_boardings'], errors='coerce')
average_golden_gate_park_shuttle_ridership = compute_averages(
    df=total_golden_gate_park_shuttle_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_name", "day_type", "stop_id"]
)


average_golden_gate_park_shuttle_ridership["organization_name"] = "Golden Gate Park Shuttle"

average_golden_gate_park_shuttle_ridership[['start_date','end_date']] = [golden_gate_park_shuttle_ridership['start_date'].min(), 
                                                                         golden_gate_park_shuttle_ridership['end_date'].max()]
golden_gate_parkshuttle_final = standardize_columns(average_golden_gate_park_shuttle_ridership, master_cols)

golden_gate_parkshuttle_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
21,Golden Gate Park Shuttle,None,None,7613,Conservatory of Flowers WB,None,None,Saturday,20.730769,None,2024-07-01,2025-06-30
37,Golden Gate Park Shuttle,None,None,7606,Music Concourse,None,None,Sunday,5.307692,None,2024-07-01,2025-06-30
27,Golden Gate Park Shuttle,None,None,7618,Haight/Stanyan,None,None,Saturday,16.403846,None,2024-07-01,2025-06-30
44,Golden Gate Park Shuttle,None,None,7603,Rose Garden WB,None,None,Weekday,9.157088,None,2024-07-01,2025-06-30
20,Golden Gate Park Shuttle,None,None,7612,Conservatory of Flowers EB,None,None,Weekday,6.616858,None,2024-07-01,2025-06-30


In [22]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")

golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
    'ROUTE': 'route_id',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "route_id", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_golden_gate_transit_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_boardings'], errors='coerce')
total_golden_gate_transit_ridership['daily_alightings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_alightings'], errors='coerce')

average_golden_gate_transit_ridership = compute_averages(
    df=total_golden_gate_transit_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", "stop_name", "day_type"]
)

average_golden_gate_transit_ridership["organization_name"] = "Golden Gate Transit"
average_golden_gate_transit_ridership[['start_date','end_date']] = [golden_gate_transit_ridership['start_date'].min(), 
                                                                         golden_gate_transit_ridership['end_date'].max()]

golden_gate_transit_final = standardize_columns(average_golden_gate_transit_ridership, master_cols)

golden_gate_transit_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1416,Golden Gate Transit,580,None,42192,Cutting Blvd & S 41st St,None,None,Saturday,4.000000,1.000000,2025-09-01,2025-09-30
349,Golden Gate Transit,114,None,40049,Battery St & Jackson St,None,None,Weekday,0.380952,6.142857,2025-09-01,2025-09-30
1150,Golden Gate Transit,164,None,80023,VTP 101 SB @ San Antonio Rd,None,None,Weekday,0.000000,0.000000,2025-09-01,2025-09-30
576,Golden Gate Transit,130,None,40455,Medway Rd & Mill St,None,None,Sunday,11.000000,0.000000,2025-09-01,2025-09-30
135,Golden Gate Transit,101,None,40951,Piner Rd & Range Ave,None,None,Saturday,0.250000,6.500000,2025-09-01,2025-09-30


In [23]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "Route", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Boardings", "sum"),
        average_daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
    'Route': 'route_id',
})

total_long_beach_transit_ridership["organization_name"] = "Long Beach Transit"

total_long_beach_transit_ridership[['start_date','end_date']] = [long_beach_transit_ridership['start_date'].min(), 
                                                                         long_beach_transit_ridership['end_date'].max()]

long_beach_transit_final = standardize_columns(total_long_beach_transit_ridership, master_cols)

long_beach_transit_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3997,Long Beach Transit,94,None,1108,7th & Ximeno NW,None,None,Sunday,18.949394,3.593663,2024-07-01,2025-06-30
1703,Long Beach Transit,112,None,1431,Clark & Spring NE,None,None,Saturday,2.796844,5.590551,2024-07-01,2025-06-30
2651,Long Beach Transit,191,None,157,Magnolia & Anaheim SW,None,None,Saturday,26.011292,47.733669,2024-07-01,2025-06-30
8176,Long Beach Transit,172,None,1692,PCH & Walnut NW,None,None,Weekday,22.176412,45.707463,2024-07-01,2025-06-30
2662,Long Beach Transit,191,None,1546,Santa Fe & Dominguez NE,None,None,Saturday,0.099813,3.291992,2024-07-01,2025-06-30


In [42]:
octa_ridership = octa_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Route': 'route_name'

})

octa_ridership["stop_name"] = octa_ridership["stop_name"].str.split("-", n=1).str[1]
octa_ridership["route_name"] = octa_ridership["route_name"].str.split("-", n=1).str[1]

group_cols = [
    "day_type", "route_name", "stop_id", "route_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
octa_ridership_sum = (octa_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("APC Boarding", "sum"),
        average_daily_alightings=("APC Alighting", "sum")
    )
)

octa_ridership_sum.loc[
    octa_ridership_sum['day_type'].str.strip().str.lower() == "weekday",
    'day_type'
] = "Weekday"

octa_ridership_sum["organization_name"] = "Orange County Transportation Authority"

octa_ridership_sum[['start_date','end_date']] = [octa_ridership['start_date'].min(), 
                                                                         octa_ridership['end_date'].max()]

octa_final = standardize_columns(octa_ridership_sum, master_cols)

octa_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
455,Orange County Transportation Authority,123,Anaheim to Huntington Beach,3448,VALLEY VIEW-CORPORATE,None,None,Weekday,0,3,2025-02-04,2025-02-04
990,Orange County Transportation Authority,86,Costa Mesa - Mission Viejo,3485,MAIN-RED HILL,None,None,Weekday,1,8,2025-02-04,2025-02-04
5057,Orange County Transportation Authority,72,Sunset Beach - Tustin,8135,TUSTIN RANCH-WARNER,None,None,Weekday,3,29,2025-02-04,2025-02-04
3095,Orange County Transportation Authority,89,Lake Forest - Laguna Beach,3874,LAGUNA CANYON-STAN OAKS,None,None,Weekday,12,12,2025-02-04,2025-02-04
5561,Orange County Transportation Authority,401,iShuttle Route B to Irvine Business Complex,8313,JAMBOREE-BECKMAN,None,None,Weekday,3,0,2025-02-04,2025-02-04


In [25]:
omni_trans_ridership= omni_trans_ridership.rename(columns={
    'Route': 'route_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'average_daily_boardings',
    'avg_alightings': 'average_daily_alightings',
    'Stop Id': 'stop_id'
})

omni_trans_ridership["organization_name"] = "Omnitrans"

omni_trans_ridership_filtered = omni_trans_ridership[['route_id', 'stop_name', 'stop_id',  'average_daily_boardings', 'average_daily_alightings', 'organization_name', 'day_type']]

omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
    lambda x: None if pd.isna(x) else int(x)
)

omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(), 
                                                                         omni_trans_ridership['end_date'].max()]
omni_final = standardize_columns(omni_trans_ridership_filtered, master_cols)

omni_final.sample(5)

/tmp/ipykernel_1813/1053658925.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
/tmp/ipykernel_1813/1053658925.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(),
/tmp/ipykernel_1813/1053658925.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
4554,Omnitrans,84,None,5942.0,Riverside @ Magnolia,None,None,all,0.213699,0.241096,2023-07-01,2026-06-30
2109,Omnitrans,10,None,6287.0,Baseline @ Madison,None,None,all,0.928767,1.230137,2023-07-01,2026-06-30
777,Omnitrans,19,None,5227.0,San Bernardino @ Linden,None,None,all,0.986339,0.756831,2023-07-01,2026-06-30
3883,Omnitrans,15,None,NaN,LUGONIA @ INDIANA CT,None,None,all,1.101370,0.536986,2023-07-01,2026-06-30
645,Omnitrans,15,None,6503.0,Merrill @ Spruce,None,None,all,1.183060,1.188525,2023-07-01,2026-06-30


In [26]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "Route", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_id'

})

In [27]:
total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_riverside_transit_ridership['daily_boardings'] = pd.to_numeric(total_riverside_transit_ridership['daily_boardings'], errors='coerce')
average_riverside_transit_ridership = compute_averages(
    df=total_riverside_transit_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["route_id", "stop_id", "day_type"]
)

average_riverside_transit_ridership["organization_name"] = "Riverside Transit"

average_riverside_transit_ridership[['start_date','end_date']] = [riverside_transit_ridership['start_date'].min(), 
                                                                         riverside_transit_ridership['end_date'].max()]

riverside_final = standardize_columns(average_riverside_transit_ridership, master_cols)

riverside_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
5166,Riverside Transit,23,None,1340,None,None,None,Weekday,1.000000,None,2025-01-01,2025-10-31
1520,Riverside Transit,10,None,1415,None,None,None,Saturday,1.000000,None,2025-01-01,2025-10-31
2788,Riverside Transit,13,None,2212,None,None,None,Weekday,11.284946,None,2025-01-01,2025-10-31
3541,Riverside Transit,15,None,1864,None,None,None,Weekday,1.406593,None,2025-01-01,2025-10-31
1030,Riverside Transit,8,None,1368,None,None,None,Weekday,2.383562,None,2025-01-01,2025-10-31


In [38]:
sacrt_bus_ridership = sacrt_bus_ridership.rename(columns={
    'route_long_name': 'route_name',
})

group_cols = [
    "stop_id", "stop_lat", "stop_lon", "stop_name", "route_name", "route_id", "day_type"]

# Combining across directions
sacrt_bus_ridership_sum = (sacrt_bus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("average_boardings", "sum"),
        average_daily_alightings=("average_alightings", "sum")
    )
)

sacrt_bus_ridership_sum.loc[
    sacrt_bus_ridership_sum['day_type'].str.strip().str.lower() == "weekday",
    'day_type'
] = "Weekday"

# Replace day_type
sacrt_bus_ridership_sum["day_type"] = sacrt_bus_ridership_sum["day_type"].replace("daily", "all")

sacrt_bus_ridership_sum["organization_name"] = "SacRT Bus"

sacrt_bus_ridership_sum[['start_date','end_date']] = [sacrt_bus_ridership['start_date'].min(), 
                                                                         sacrt_bus_ridership['end_date'].max()]

sacrt_bus_final = standardize_columns(sacrt_bus_ridership_sum, master_cols)

sacrt_bus_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2360,SacRT Bus,081,FLORIN,2361,FLORIN RD & 17TH ST (EB),38.495651,-121.499031,all,5,4,2023-09-01,2023-12-31
770,SacRT Bus,176,CordoVan - Anatolia-Kavala Ranch,663,NORTH MATHER BLVD & SPOTO DR (WB),38.569225,-121.279058,Weekday,0,0,2023-09-01,2023-12-31
3737,SacRT Bus,255,LA RIVIERA - COLLEGE GREENS,5333,WISSEMANN DR & BRIGHAM WAY (NB),38.554430,-121.381575,Weekday,0,1,2023-09-01,2023-12-31
2180,SacRT Bus,252,FREEPORT - FRUITRIDGE - ML KING,2235,FREEPORT BLVD & SUTTERVILLE RD (NB),38.538648,-121.492595,Weekday,0,0,2023-09-01,2023-12-31
696,SacRT Bus,015,DEL PASO HEIGHTS,574,WINTERS ST & GRAND AVE (SB),38.637245,-121.411606,all,2,4,2023-09-01,2023-12-31


In [29]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Route': 'route_id',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
samtrans_ridership['daily_boardings'] = pd.to_numeric(samtrans_ridership['daily_boardings'], errors='coerce')
samtrans_ridership['daily_alightings'] = pd.to_numeric(samtrans_ridership['daily_alightings'], errors='coerce')

average_samtrans_ridership = compute_averages(
    df=samtrans_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"]
)

average_samtrans_ridership["organization_name"] = "Samtrans"

average_samtrans_ridership[['start_date','end_date']] = [samtrans_ridership['start_date'].min(), 
                                                                         samtrans_ridership['end_date'].max()]

samtrans_final = standardize_columns(average_samtrans_ridership, master_cols)

samtrans_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2425,Samtrans,260,None,342036,Continentals Way & Cipriani Blvd,37.512203,-122.303749,Sunday,2.6,5.2,2025-08-01,2025-08-31
6437,Samtrans,ECRO,None,334076,El Camino & Arroyo Dr,37.657230,-122.435729,Sunday,1.0,0.4,2025-08-01,2025-08-31
4956,Samtrans,40,None,335616,Riverside Dr & Sneath Ln,37.628246,-122.452323,Weekday,1.0,0.0,2025-08-01,2025-08-31
3952,Samtrans,296,None,346182,Willow Rd & Coleman Ave,37.462001,-122.159385,Saturday,5.2,13.8,2025-08-01,2025-08-31
6417,Samtrans,ECRO,None,332544,Mission St & Wellington Ave,37.706157,-122.398063,Saturday,6.0,13.6,2025-08-01,2025-08-31


In [30]:
group_cols = [
    "Route", "Route Name", "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Average On", "sum"),
        average_daily_alightings=("Average Off", "sum")
    )
)

total_sdmts_ridership["Route Name"] = total_sdmts_ridership["Route Name"].str.split(":", n=1).str[1]

total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_id',
    'Route Name': 'route_name',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership["organization_name"] = "SDMTS"

total_sdmts_ridership[['start_date','end_date']] = [sdmts_ridership['start_date'].min(), 
                                                                         sdmts_ridership['end_date'].max()]

sdmts_final = standardize_columns(total_sdmts_ridership, master_cols)

sdmts_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2716,SDMTS,13,Kaiser Hospital-24th St TC,11727,Fairmount Av & Dwight St,None,None,Sunday,10.011474,13.952512,2024-09-01,2025-01-25
4551,SDMTS,43,Balboa Av TC-Kearny Mesa,89002,Balboa Avenue Transit Center,None,None,Weekday,346.974654,308.454376,2024-09-01,2025-01-25
102,SDMTS,1,Fashion Valley-La Mesa,10659,El Cajon Bl & 58th St,None,None,Sunday,0.905952,10.209106,2024-09-01,2025-01-25
1385,SDMTS,6,North Park-Fashion Valley,12824,30th St & University Av,None,None,Weekday,131.229350,0.016129,2024-09-01,2025-01-25
11823,SDMTS,933,Iris TC Loop-Imperial Beach Counterclock,60193,Palm Av & Desty St,None,None,Saturday,0.760088,0.696637,2024-09-01,2025-01-25


In [31]:
santa_cruz_metro_ridership = santa_cruz_metro_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Boardings': 'average_daily_boardings',
    'Alightings': 'average_daily_alightings'    
})

santa_cruz_metro_ridership["organization_name"] = "Santa Cruz Metro"

santa_cruz_metro_ridership[['start_date','end_date']] = [santa_cruz_metro_ridership['start_date'].min(), 
                                                                         santa_cruz_metro_ridership['end_date'].max()]

santa_cruz_final = standardize_columns(santa_cruz_metro_ridership, master_cols)

santa_cruz_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
624,Santa Cruz Metro,None,None,1748,Scotts Valley Dr + Terrace View Dr,None,None,all,0.852055,2.328767,2024-07-01,2025-06-30
278,Santa Cruz Metro,None,None,1451,Freedom Blvd (Rider Apples),None,None,all,0.115068,0.438356,2024-07-01,2025-06-30
158,Santa Cruz Metro,None,None,1315,Center Ave + El Camino Del Mar,None,None,all,0.008219,0.024658,2024-07-01,2025-06-30
176,Santa Cruz Metro,None,None,1332,Clubhouse Dr + Murray Ave,None,None,all,0.073973,0.024658,2024-07-01,2025-06-30
418,Santa Cruz Metro,None,None,1554,Hwy 9 + Clear Creek Rd,None,None,all,10.008219,2.178082,2024-07-01,2025-06-30


In [32]:
sbmtd_ridership = sbmtd_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'daily_boardings',
    'avg_ridership': 'daily_alightings'    
})


sbmtd_ridership["n_days"] = sbmtd_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 12 months
averaged_sbmtd_ridership = (
    sbmtd_ridership
    .groupby(['stop_id', 'stop_name', 'day_type'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['Ridership by Stop_Boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['Ridership by Stop_Alightings'], weights=x['n_days'])
    }))
    .reset_index()
)



averaged_sbmtd_ridership["organization_name"] = "SBMTD"

averaged_sbmtd_ridership[['start_date','end_date']] = [sbmtd_ridership['start_date'].min(), 
                                                                         sbmtd_ridership['end_date'].max()]

sbmtd_final = standardize_columns(averaged_sbmtd_ridership, master_cols)

sbmtd_final.sample(5)

/tmp/ipykernel_1813/1734702637.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
45,SBMTD,None,None,59,Milpas & Cota,None,None,all,567.531507,1277.830137,2024-11-01,2025-10-31
552,SBMTD,None,None,677,Rhoads & Ripley,None,None,all,10.032877,6.821918,2024-11-01,2025-10-31
556,SBMTD,None,None,681,Las Positas & Modoc,None,None,all,15.989041,0.169863,2024-11-01,2025-10-31
329,SBMTD,None,None,390,Cathedral Oaks & Kellogg,None,None,all,57.104110,5.186301,2024-11-01,2025-10-31
338,SBMTD,None,None,399,Cathedral Oaks & Santa Marguerita,None,None,all,23.789041,8.909589,2024-11-01,2025-10-31


In [45]:
all_ridership = pd.concat([
    bart_final,
    big_blue_bus_ridership_final,
    caltrain_ridership_final,
    culver_citybus_final,
    foothill_final,
    fresno_final,
    golden_coast_transit_final,
    golden_gate_parkshuttle_final,
    golden_gate_transit_final,
    long_beach_transit_final,
    octa_final,
    omni_final,
    riverside_final,
    sacrt_bus_final,
    samtrans_final,
    sdmts_final,
    santa_cruz_final,
    sbmtd_final,
], ignore_index=True)

/tmp/ipykernel_1813/1180593662.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ridership = pd.concat([


In [46]:
all_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71703 entries, 0 to 71702
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   organization_name         71703 non-null  object        
 1   route_id                  65173 non-null  object        
 2   route_name                27000 non-null  object        
 3   stop_id                   71293 non-null  object        
 4   stop_name                 56894 non-null  object        
 5   stop_lat                  21395 non-null  float64       
 6   stop_lon                  21395 non-null  float64       
 7   day_type                  71703 non-null  object        
 8   average_daily_boardings   71703 non-null  float64       
 9   average_daily_alightings  63461 non-null  float64       
 10  start_date                71703 non-null  datetime64[ns]
 11  end_date                  71703 non-null  datetime64[ns]
dtypes: datetime64[ns](

In [46]:
all_ridership.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/ridership_all.csv", index=False)